In [12]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [13]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=False)

In [14]:
from hummingbot.strategy_v2.utils.distributions import Distributions
from controllers.market_making.pmm_dynamic import PMMDynamicControllerConfig
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal

# Controller configuration
connector_name = "binance"
trading_pair = "POL-USDT"
total_amount_quote = 1000 # Total amount in quote currency (USDT)
take_profit = 0.02
stop_loss = 0.01
trailing_stop_activation_price = 0.015
trailing_stop_trailing_delta = 0.07
time_limit = 60 * 60 * 2
executor_refresh_time = 60 * 6
cooldown_time = 600

# order book levels
sell_order_book_levels = 4
buy_order_book_levels = 4

# Backtesting configuration
start = int(datetime.datetime(2025, 5, 24).timestamp())
end = int(datetime.datetime(2025, 5, 25).timestamp())
backtesting_resolution = "1m"

config = PMMDynamicControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    candles_connector=connector_name,
    candles_trading_pair=trading_pair, 
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(
        activation_price=Decimal(trailing_stop_activation_price),
        trailing_delta=Decimal(trailing_stop_trailing_delta)
    ),
    time_limit=time_limit,
    executor_refresh_time=executor_refresh_time,
    cooldown_time=cooldown_time,
    buy_spreads=[0.5, 1.0, 1.5, 2.0],
    sell_spreads=[0.5, 1.0, 1.5, 2.0],
    buy_amounts_pct=[Decimal(0.1), Decimal(0.2), Decimal(0.3), Decimal(0.4)],
    sell_amounts_pct=[Decimal(0.1), Decimal(0.2), Decimal(0.3), Decimal(0.4)],
)

In [15]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution)

In [16]:
backtesting_result.controller_config

PMMDynamicControllerConfig(id=None, controller_name='pmm_dynamic', controller_type='market_making', total_amount_quote=Decimal('1000'), manual_kill_switch=False, candles_config=[CandlesConfig(connector='binance', trading_pair='POL-USDT', interval='3m', max_records=142)], connector_name='binance', trading_pair='POL-USDT', buy_spreads=[0.5, 1.0, 1.5, 2.0], sell_spreads=[0.5, 1.0, 1.5, 2.0], buy_amounts_pct=[Decimal('0.1000000000000000055511151231257827021181583404541015625'), Decimal('0.200000000000000011102230246251565404236316680908203125'), Decimal('0.299999999999999988897769753748434595763683319091796875'), Decimal('0.40000000000000002220446049250313080847263336181640625')], sell_amounts_pct=[Decimal('0.1000000000000000055511151231257827021181583404541015625'), Decimal('0.200000000000000011102230246251565404236316680908203125'), Decimal('0.299999999999999988897769753748434595763683319091796875'), Decimal('0.40000000000000002220446049250313080847263336181640625')], executor_refresh_ti

In [17]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()



Net PNL: $2.66 (0.27%) | Max Drawdown: $-5.51 (-0.55%)
Total Volume ($): 14900.56 | Sharpe Ratio: 0.26 | Profit Factor: 1.15
Total Executors: 510 | Accuracy Long: 0.32 | Accuracy Short: 0.70
Close Types: Take Profit: 0 | Stop Loss: 7 | Time Limit: 63 |
             Trailing Stop: 0 | Early Stop: 440



In [18]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [19]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
# fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [20]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()
